# Annotations

## Links

#### Create annotations:
 - [Yolov5 documentation](https://docs.ultralytics.com/yolov5/tutorials/train_custom_data/#12-create-labels_1)
 - Satellite image annotation [GitHub](https://github.com/satellite-image-deep-learning/annotation)
   - [IRIS](https://github.com/ESA-PhiLab/iris)
   - [Kili](https://kili-technology.com/data-labeling/best-geospatial-annotation-tool-what-to-look-for-in-software#finding-the-right-geospatial-annotation-tool)
 - Python [split_raster](https://github.com/cuicaihao/split_raster) lib

#### Split Raster(s)

 - [xarray doc](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.to_numpy.html)

#### Split Raster(s)

 - [gdal_translate](https://gdal.org/programs/gdal_translate.html) documentation
 - [test GDAL Translate](https://svn.osgeo.org/gdal/trunk/autotest/utilities/test_gdal_translate_lib.py) code
 - [gdaltest.py](https://github.com/OSGeo/gdal/blob/master/autotest/pymod/gdaltest.py) on GitHub

## Load

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# %load import.py
# 
import numpy as np
import pandas as pd

#
from sentinelsat import SentinelAPI, read_geojson, geojson_to_wkt
from datetime import date

#
import rioxarray
import geopandas as gpd
import rasterio as rio

#
from matplotlib import pyplot
from rasterio.plot import show

#
from sqlalchemy import create_engine # query PostGIS
from sqlalchemy import inspect

#
import osmnx as ox

#
from shapely.geometry import Polygon, box
import shapely.ops as so

#
import json

#
import os
from pathlib import Path
import fnmatch
import glob

#
from osgeo import gdal

#
import random

#
import shutil

#
from tqdm import tqdm
import time

#
import folium
from folium import plugins

## Split Raster(s)

#### Load GeoDF

In [ ]:
sel_prod_gdf = gpd.read_file("sel_products.geojson")

In [ ]:
prod_id = sel_prod_gdf["title"][0]
prod_id

#### Reference raster

In [ ]:
ref_ras_file = "./data/S2A_MSIL1C_20230427T095031_N0509_R079_T33TVF_20230427T115004.SAFE/GRANULE/L1C_T33TVF_A040974_20230427T095813/IMG_DATA/T33TVF_20230427T095031_TCI.jp2"

In [ ]:
ref_ras = rio.open( ref_ras_file )

In [ ]:
bbox = ref_ras.bounds
bbox

In [ ]:
ref_ras.shape

#### Procedure to Split Raster

##### DEF

In [ ]:
tileSize = 32*15
nTilesX = int(ref_ras.shape[0] / tileSize)
nTilesY = int(ref_ras.shape[1] / tileSize)
print("Tile size %d x %d." % (tileSize,tileSize))
print("Grid size %d x %d." % (nTilesX,nTilesY))

The condition $Tot \;\%\; tileSize = 0$ must be verified, where <br><br>
$Tot = (Nte \times tileSize) - (Nte-1)\times olSize$, where Nte = nTiles[X|Y] +1 

The condition above can be written as <br><br>
$(Nte \times tileSize) - (Nte-1) \times olSize = gridSize$, which for the problem at hand becomes: <br><br>
$olSize = \frac{(Nte \,\times\, tileSize) \,-\, gridSize}{(Nte\,-\,1)}$

In [ ]:
def find_olSize(nTiles,tileSize,gridSize):
    Found = False
    for t in range(30):
        if t > 0:
            # Nte: `Ntiles Extended` to accommodate the overlapping zone
            Nte = nTiles + t
            #print('nTiles: ',Nte)
            Mod = ( (Nte*tileSize) - gridSize ) % (Nte-1)
            if Mod==0:
                Found = True
                olSize = ( (Nte*tileSize) - gridSize ) / (Nte-1)
                #print( '  olSize = %d (found at Nte=%d)' % (olSize,Nte) )
                break

    if not Found:
        return(None,None)
    else:
        return(olSize,Nte)

In [ ]:
olSizeX,NteX = find_olSize(nTilesX,tileSize,ref_ras.shape[0])
olSizeY,NteY = find_olSize(nTilesY,tileSize,ref_ras.shape[0])
print("olSizeX=%d , NteX=%d , isCorrect=%d" % (olSizeX,NteX,bool((NteX * tileSize - (NteX-1)*olSizeX) - ref_ras.shape[0] == 0) ))
print("olSizeY=%d , NteY=%d , isCorrect=%d" % (olSizeY,NteY,bool((NteY * tileSize - (NteY-1)*olSizeY) - ref_ras.shape[1] == 0) ))

In [ ]:
def raster_tile_overlap(ras_file,oPath,tile_base_name,tSx,tSy,NteX,NteY,olSx,olSy):

    dso = gdal.Open( ras_file )
    # ==== GDAL INFO
    band  = dso.GetRasterBand(1)
    xsize = band.XSize
    ysize = band.YSize
    print( "Xsize             :  " + str(xsize) )
    print( "Ysize             :  " + str(ysize) )
    print( "Output path       :  " + oPath )
    print( "Tile base name    :  " + tile_base_name + "__i_j.tif" )
    print( "rmn, rmx          :  " + "row min, row max" )
    print( "cmn, cmx          :  " + "col min, col max" )
    
    # srcWin = [i,j,tile_size_x,tile_size_y]
    
    ii=0
    for tX in range(NteX):
        ii = ii + 1
        
        i = tX*(tSx - olSx)

        jj=0
        for tY in range(NteY):
            jj = jj + 1
            
            j = tY*(tSy - olSy)
            
            tile_name_ij = tile_base_name + "__" + \
                            str(ii).rjust(3,'0') + "_" + \
                            str(jj).rjust(3,'0') + ".tif"
            
            print( "i: %03d j: %03d  -  rmn: %5d rmx: %5d cmn: %5d cmx: %5d" % (ii, jj, i, i+tSx, j, j+tSy) )

            ds = gdal.Translate(oPath + tile_name_ij,
                                dso,
                                creationOptions = ['COMPRESS=LZW','NUM_THREADS=ALL_CPUS','INTERLEAVE=BAND'],
                                srcWin = [i,j,tSx,tSy]
                               )

            if ds is None:
                return 'fail::isNotNone - ' + str(i) + ' - ' + str(j)
            else:
                ds = None
                
    dso = None

    return 'success'

In [ ]:
def raster_tile(ras_file,oPath,tile_base_name,tile_size_x,tile_size_y):

    dso = gdal.Open( ras_file )
    # ==== GDAL INFO
    band  = dso.GetRasterBand(1)
    xsize = band.XSize
    ysize = band.YSize
    print( "Xsize             :  " + str(xsize) )
    print( "Ysize             :  " + str(ysize) )
    print( "Output path       :  " + oPath )
    print( "Tile base name    :  " + tile_base_name + "__i_j.tif" )
    print( "rmn, rmx          :  " + "row min, row max" )
    print( "cmn, cmx          :  " + "col min, col max" )
    
    ii=0
    for i in range(0, xsize, tile_size_x):
        ii = ii + 1
        jj=0
        tile_size_x_end = min( tile_size_x, xsize - (ii-1)*tile_size_x )

        for j in range(0, ysize, tile_size_y):
            jj = jj + 1
            tile_size_y_end = min( tile_size_y, ysize - (jj-1)*tile_size_y )
            
            tile_name_ij = tile_base_name + "__" + \
                            str(ii).rjust(3,'0') + "_" + \
                            str(jj).rjust(3,'0') + ".tif"
            
            print( "i: %03d j: %03d  -  rmn: %5d rmx: %5d cmn: %5d cmx: %5d" % (ii, jj, i, i+tile_size_x_end, j, j+tile_size_y_end) )

            ds = gdal.Translate(oPath + tile_name_ij,
                                dso,
                                creationOptions = ['COMPRESS=LZW','NUM_THREADS=ALL_CPUS','INTERLEAVE=BAND'],
                                srcWin = [i,j,tile_size_x_end,tile_size_y_end]
                               )

            if ds is None:
                return 'fail::isNotNone - ' + str(i) + ' - ' + str(j)
            else:
                ds = None
                
    dso = None

    return 'success'

##### Run

In [ ]:
dir_annotation_ras = "annotations/" + str(tileSize) + "x" + str(tileSize) + "/" + prod_id + "/"
dir_annotation_ras

In [ ]:
if not os.path.exists( dir_annotation_ras ):
    print("Creating directory %s" % dir_annotation_ras)
    os.makedirs(dir_annotation_ras)

In [ ]:
#raster_tile(ref_ras_file,dir_annotation_ras,prod_id,610,610)

In [ ]:
raster_tile_overlap( ref_ras_file , dir_annotation_ras , prod_id , tileSize,tileSize, NteX,NteY, olSizeX,olSizeY)

#### Plot notes

In [ ]:
i=13
j=12
nr = rioxarray.open_rasterio( dir_annotation_ras + prod_id + "__" + str(i).rjust(3,'0') + "_" + str(j).rjust(3,'0') + ".tif")
nr.plot.imshow()

In [ ]:
nr

In [ ]:
src = rio.open( dir_annotation_ras + "S2A_MSIL1C_20230427T095031_N0509_R079_T33TVF_20230427T115004__001_001.tif" )

In [ ]:
pyplot.imshow(src.read(1), cmap='pink')

In [ ]:
fig, (axr, axg, axb) = pyplot.subplots(1,3, figsize=(21,7))

show((src, 1), ax=axr, cmap='Reds', title='red channel')
show((src, 2), ax=axg, cmap='Greens', title='green channel')
show((src, 3), ax=axb, cmap='Blues', title='blue channel')

pyplot.show()

In [ ]:
masked = rio.open( dir_annotation_ras + "S2A_MSIL1C_20230427T095031_N0509_R079_T33TVF_20230427T115004__001_001.tif" )
pyplot.imshow(masked.read(1), cmap='pink')

## Split Polygon(s)

 - [Yolo8 documentation about polygon annotations](https://docs.ultralytics.com/datasets/segment/)

#### Dev code

*PREMISE*<br>
I split the raster (110km x 110km) in square tiles of size 610 pixels.<br>
I inserted all the downloaded geojson OSM tags (only building for now) in postGIS.<br>

*OBJECTIVE*<br>
I have to
 - load the bbox of any raster tile
 - intersect in postGIS the ['Polygon','MultiPolygon'] geometries
 - for each geometry, write the correspondent row as `label_id x_1 y_1 x_2 y_2 x_3 y_3 ...`, where
   - label_id: numeric from 0
   - a point Pi(x_i , y_i) has relative coordinates where
     - xmin : left   bbox border =0
     - ymin : top    bbox border =0
     - xmax : right  bbox border =1
     - ymax : bottom bbox border =1

##### Select GRANULE

In [ ]:
sel_prod_gdf = gpd.read_file("sel_products.geojson")

In [ ]:
prod_id = sel_prod_gdf["title"][0]
prod_id

In [ ]:
dir_annotation_ras = "annotations/" + str(tileSize) + "x" + str(tileSize) + "/" + prod_id + "/"
dir_annotation_ras

In [ ]:
import glob

In [ ]:
f = glob.glob( dir_annotation_ras + '*.tif' )
len(f)

In [ ]:
i=321

In [ ]:
f[i]

In [ ]:
f[i].split("/")[3]

In [ ]:
src = rio.open(f[i])

In [ ]:
type(src)

In [ ]:
src

In [ ]:
show(src)

##### GRANULE BBox

In [ ]:
xn,yn,xx,yx = src.bounds
print('Xmin: %8.3f, Xmax: %8.3f, Ymin: %8.3f, Ymax: %8.3f' % (xn,xx,yn,yx) )

In [ ]:
dx = xx-xn
dx

In [ ]:
dy = yx-yn
dy

In [ ]:
bb_pol = Polygon([ [xn,yx],[xx,yx],[xx,yn],[xn,yn],[xn,yx] ])
print(type(bb_pol))
print(bb_pol)
bb_pol

In [ ]:
bb_gdf = gpd.GeoDataFrame(index=[0], crs='epsg:32633', geometry=[bb_pol])
print(bb_gdf.geometry[0])

In [ ]:
pol_str = "POLYGON((" + str(xn) + " " + str(yx) + ", " + \
                        str(xx) + " " + str(yx) + ", " + \
                        str(xx) + " " + str(yn) + ", " + \
                        str(xn) + " " + str(yn) + ", " + \
                        str(xn) + " " + str(yx) + "))"
print(pol_str)

In [ ]:
print(bb_gdf.to_crs(4326).geometry[0])

##### Folium map

https://www.linkedin.com/pulse/visualize-dem-interactive-map-chonghua-yin/?trk=related_artice_Visualize%20DEM%20in%20An%20Interactive%20Map_article-card_title

In [ ]:
m = folium.Map([40, 14], zoom_start=7, tiles='cartodbpositron')
folium.GeoJson('naples_metropolytan.geojson').add_to(m)
folium.LatLngPopup().add_to(m)
#m.fit_bounds([[xn,yn],[xx,yx]])
m

In [ ]:
i=13
j=12
fil = dir_annotation_ras + prod_id + "__" + str(i).rjust(3,'0') + "_" + str(j).rjust(3,'0') + ".tif"

In [ ]:
# read xarray data
xad = rioxarray.open_rasterio( fil )
print(type(xad))
# convert to numpy array:
npa = xad.to_numpy()
print(type(npa))

In [ ]:
xad

In [ ]:
src = rio.open( fil )
xn,yn,xx,yx = src.bounds
xc = (xn + xx)/2
yc = (yn + yx)/2

In [ ]:
m = folium.Map( location = [40.7, 14], 
                zoom_start=9,
               tiles='cartodbpositron' #tiles="Stamen Terrain"
              )

folium.GeoJson('naples_metropolytan.geojson').add_to(m)
folium.LatLngPopup().add_to(m)


# Overlay the image
folium.raster_layers.ImageOverlay(fil,
                                  [[yn, xn], [yx, xx]],
                                  opacity=0.7).add_to(m)

html_file = 'raster.html'
m.save(html_file)
IFrame(src=html_file, width=900, height=400)

##### PSQL query to get geometries within the GRANULE BBox

In [ ]:
qry = """
SELECT ST_Intersection(ST_Transform(a.geometry,32633), b.geometry) as geometry
    FROM public.buildings a JOIN ST_GeomFromText('""" + pol_str + """',32633) as b
    ON ST_Intersects(ST_Transform(a.geometry,32633), b.geometry)
"""
print(qry)

In [ ]:
from impervious.config import pg_url  # connessione da .env
db_connection_url = pg_url()
con = create_engine(db_connection_url)

In [ ]:
res = gpd.read_postgis(qry,con,geom_col='geometry')

In [ ]:
type(res)

In [ ]:
geoSeries = res.geometry
type(geoSeries)

In [ ]:
res.shape

In [ ]:
res.head(2)

In [ ]:
res.geometry[1]

In [ ]:
res.crs

##### remove duplicate geometries
 - Check out [this interesting blog](https://ml-gis-service.com/index.php/2021/09/24/toolbox-drop-duplicated-geometries-from-geodataframe/)
 - Read also [differences between GeoSeries and GeoDataframes](https://geopandas.org/en/stable/docs/user_guide/data_structures.html)
 
 Do not use **gdf.drop_duplicates** since it works without taking care of the geometries.<br>
 Use the DEF function reported below.

In [ ]:
cleaned = res.drop_duplicates('geometry')

In [ ]:
cleaned.shape

In [ ]:
res.shape[0] - cleaned.shape[0]

In [ ]:
def get_unique_geom_indexes(geoseries: gpd.GeoSeries):
    """
    Function modified from the original one above.
            
    INPUT:
    
    :param geoseries: (gpd.GeoSeries)
    
    OUTPUT:
    
    :returns: (list)
    """
    
    indexes_to_skip = []
    processed_indexes = []
    
    for index, geom in geoseries.items():
        if index not in indexes_to_skip:
            processed_indexes.append(index)
            indexes_to_skip.append(index)
            for other_index, other_geom in geoseries.items():
                if other_index in indexes_to_skip:
                    pass
                else:
                    if geom.equals(other_geom):
                        indexes_to_skip.append(other_index)
                    else:
                        pass
    return processed_indexes

In [ ]:
def drop_duplicated_geometries(geoseries: gpd.GeoSeries):
    """
    Function drops duplicated geometries from a geoseries. It works as follow:
    
        1. Take record from the dataset. Check it's index against list of indexes-to-skip.
        If it's not there then move to the next step.
        2. Store record's index in the list of processed indexes (to re-create geoseries without duplicates)
        and in the list of indexes-to-skip.
        3. Compare this record to all other records. If any of them is a duplicate then store its index in
        the indexes-to-skip.
        4. If all records are checked then re-create dataframe without duplicates based on the list
        of processed indexes.
        
    INPUT:
    
    :param geoseries: (gpd.GeoSeries)
    
    OUTPUT:
    
    :returns: (gpd.Geoseries)
    """
    
    indexes_to_skip = []
    processed_indexes = []
    
    for index, geom in geoseries.items():
        if index not in indexes_to_skip:
            processed_indexes.append(index)
            indexes_to_skip.append(index)
            for other_index, other_geom in geoseries.items():
                if other_index in indexes_to_skip:
                    pass
                else:
                    if geom.equals(other_geom):
                        indexes_to_skip.append(other_index)
                    else:
                        pass
    output_gs = geoseries[processed_indexes].copy()
    return output_gs

In [ ]:
gs_cleaned = drop_duplicated_geometries(res.geometry)

In [ ]:
gs_cleaned.shape

In [ ]:
gs_unique = get_unique_geom_indexes(res.geometry)

In [ ]:
len(gs_unique)

In [ ]:
#res2 = res.concat([res.iloc[1],df.loc[:]]).reset_index(drop=True)
res2 = res
res2.loc[len(res2)] = res2.iloc[1]

In [ ]:
res2.shape

In [ ]:
gs_unique = get_unique_geom_indexes(res2.geometry)
len(gs_unique)

In [ ]:
res2 = res2.iloc[gs_unique]
res2.shape

##### Convert CRS of geometries

In [ ]:
r = res.to_crs(32633)

In [ ]:
r

##### Convert geospatial coordinates into yolo relative ones

In [ ]:
r.geometry.type.unique()

In [ ]:
r[r.geometry.type=='MultiPolygon']

In [ ]:
row=34

In [ ]:
r.geometry[row]

In [ ]:
str(r.geometry[row])

In [ ]:
r.iloc[[row]]

In [ ]:
for i, row in r.iterrows():
    if row.geometry.type=='Polygon':
        x,y = np.array(row.geometry.exterior.coords.xy)
        xr = (np.array(x) - xn) / (xx-xn)
        yr = (yx - np.array(y)) / (yx-yn)

        txt_line = '0 '
        for j in range(len(xr)-1):
            txt_line = txt_line + " " + str(xr[j]) + " " + str(yr[j])
        print(txt_line)
        
    if row.geometry.type=='MultiPolygon':
        re = row.explode()
        for g in re.geometry:
            x,y = np.array(g.exterior.coords.xy)
            xr = (np.array(x) - xn) / (xx-xn)
            yr = (yx - np.array(y)) / (yx-yn)            
            txt_line = '0 '
            for j in range(len(xr)-1):
                txt_line = txt_line + " " + str(xr[j]) + " " + str(yr[j])
            print(txt_line)

In [ ]:
xr = (np.array(x) - xn) / (xx-xn)
yr = (yx - np.array(y)) / (yx-yn)

In [ ]:
print(xr)
print(yr)

In [ ]:
txt_line = '0 '
for i in range(len(xr)-1):
    txt_line = txt_line + " " + str(xr[i]) + " " + str(yr[i])
print(txt_line)

In [ ]:
ftxt = f[2].split('.tif')[0] + '.txt'
print(ftxt)

In [ ]:
with open(ftxt,'w') as txt:
    txt.write(txt_line + "\n")

In [ ]:
# test annotations
r.to_file('annotations/test/r.geojson')

In [ ]:
x1 = np.array([0, 0, 1, 1, 0])
y1 = np.array([0, 1, 1, 0, 0])

##### explore some plots

In [ ]:
fig,ax = pyplot.subplots()

ax.plot(xr, yr, color='#6699cc', alpha=0.7,
    linewidth=3, solid_capstyle='round', zorder=2)
ax.plot(x1,y1)
ax.set_title('Polygon')

pyplot.show()

In [ ]:
fig,ax = pyplot.subplots()

ax.plot(xr, yr, color='#6699cc', alpha=0.7,
    linewidth=3, solid_capstyle='round', zorder=2)
ax.plot(x1,y1)
ax.set_title('Polygon')

pyplot.show()

In [ ]:
ftxt = f[2].split('.tif')[0] + '.txt'
with open(ftxt,'w') as txt:
    for g in r.geometry:
        #print(g)
        x,y = np.array(g.exterior.coords.xy)
        xr = (np.array(x) - xn) / (xx-xn)
        yr = (yx - np.array(y)) / (yx-yn)
        txt_line = '0 '
        for i in range(len(xr)-1):
            txt_line = txt_line + " " + str(xr[i]) + " " + str(yr[i])
        print(txt_line)
        
        txt.write(txt_line + "\n")

In [ ]:
txt_line

In [ ]:
def pol_square(xn,xx,yn,yx):
    pol_str = "POLYGON((" + str(xn) + " " + str(yx) + ", " + \
                            str(xx) + " " + str(yx) + ", " + \
                            str(xx) + " " + str(yn) + ", " + \
                            str(xn) + " " + str(yn) + ", " + \
                            str(xn) + " " + str(yx) + "))"
    return pol_str

In [ ]:
def Polygon_square(xn,xx,yn,yx):
    pol_str = Polygon([(xn,yx),
                       (xx,yx),
                       (xx,yn),
                       (xn,yn),
                    ])
    return pol_str

In [ ]:
pol_str = pol_square(0.1,0.5,1-0.1,1-0.5)

In [ ]:
box = Polygon_square(0,1,0,1)
pol_str = Polygon_square(0.1,0.5,1-0.1,1-0.5)

In [ ]:
p = gpd.GeoSeries(pol_str)
b = gpd.GeoSeries(box)
p.plot()
b.plot()
pyplot.show()

In [ ]:
polygon1 = Polygon([(0,5),
                    (1,1),
                    (3,0),
                    ])
print(polygon1)

p = gpd.GeoSeries(polygon1)
p.plot()
pyplot.show()

In [ ]:
import numpy as np
from matplotlib.path import Path
from matplotlib.patches import PathPatch
from matplotlib.collections import PatchCollection


# Plots a Polygon to pyplot `ax`
def plot_polygon(ax, poly, **kwargs):
    path = Path.make_compound_path(
        Path(np.asarray(poly.exterior.coords)[:, :2]),
        *[Path(np.asarray(ring.coords)[:, :2]) for ring in poly.interiors])

    patch = PathPatch(path, **kwargs)
    collection = PatchCollection([patch], **kwargs)
    
    ax.add_collection(collection, autolim=True)
    ax.autoscale_view()
    return collection

In [ ]:
from shapely.geometry import Polygon
import matplotlib.pyplot as plt

polygon = Polygon(shell=((0,0),(1,0),(1,1),(0,1)),
                  holes=(((0.1,0.1),(0.1,0.5),(0.5,0.5),(0.5,0.1)),
                         ((0.9,0.9),(0.9,0.5),(0.5,0.5),(0.5,0.9)))
                 )

fig, ax = pyplot.subplots()
plot_polygon(ax, polygon, facecolor='lightblue', edgecolor='red')

#### BATCH processing

In [ ]:
def drop_duplicated_geometries(geoseries: gpd.GeoSeries):
    """
    Function drops duplicated geometries from a geoseries. It works as follow:
    
        1. Take record from the dataset. Check it's index against list of indexes-to-skip.
        If it's not there then move to the next step.
        2. Store record's index in the list of processed indexes (to re-create geoseries without duplicates)
        and in the list of indexes-to-skip.
        3. Compare this record to all other records. If any of them is a duplicate then store its index in
        the indexes-to-skip.
        4. If all records are checked then re-create dataframe without duplicates based on the list
        of processed indexes.
        
    INPUT:
    
    :param geoseries: (gpd.GeoSeries)
    
    OUTPUT:
    
    :returns: (gpd.Geoseries)
    """
    
    indexes_to_skip = []
    processed_indexes = []
    
    for index, geom in geoseries.items():
        if index not in indexes_to_skip:
            processed_indexes.append(index)
            indexes_to_skip.append(index)
            for other_index, other_geom in geoseries.items():
                if other_index in indexes_to_skip:
                    pass
                else:
                    if geom.equals(other_geom):
                        indexes_to_skip.append(other_index)
                    else:
                        pass
    output_gs = geoseries[processed_indexes].copy()
    return output_gs

In [ ]:
def get_unique_geom_indexes(geoseries: gpd.GeoSeries):
    """
    Function modified from the original one above.
            
    INPUT:
    
    :param geoseries: (gpd.GeoSeries)
    
    OUTPUT:
    
    :returns: (list)
    """
    
    indexes_to_skip = []
    processed_indexes = []
    
    for index, geom in geoseries.items():
        if index not in indexes_to_skip:
            processed_indexes.append(index)
            indexes_to_skip.append(index)
            for other_index, other_geom in geoseries.items():
                if other_index in indexes_to_skip:
                    pass
                else:
                    if geom.equals(other_geom):
                        indexes_to_skip.append(other_index)
                    else:
                        pass
    return processed_indexes

In [ ]:
sel_prod_gdf = gpd.read_file("sel_products.geojson")
sel_prod_gdf.iloc[[0]]

In [ ]:
prod_id = sel_prod_gdf["title"][0]
prod_id

In [ ]:
dir_annotation_ras = "annotations/" + str(tileSize) + "x" + str(tileSize) + "/" + prod_id + "/"
dir_annotation_ras

In [ ]:
skipExistent = True

In [ ]:
ftxt = glob.glob( "annotations/480x480/S2A_MSIL1C_20230427T095031_N0509_R079_T33TVF_20230427T115004/S2A_MSIL1C_20230427T095031_N0509_R079_T33TVF_20230427T115004__001_007.txt" )
ftxt = ftxt[0]
ftxt

In [ ]:
from impervious.config import pg_url  # connessione da .env
import warnings
from shapely.errors import ShapelyDeprecationWarning
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)

db_connection_url = pg_url()
con = create_engine(db_connection_url)

i = -1
Count = 0
for f in glob.glob( dir_annotation_ras + '*.tif' ):
    ftxt = f.split('.tif')[0] + '.txt'
    
    i = i +1
    if i >= 0:
        print("%04d\t%s" % (i,f.split("/")[3]))
        
        if glob.glob( ftxt ):
            print( '  > SKIP --> %s exists!' % f.split("/")[3].replace('.tif','.txt') )
            continue

        src = rio.open(f)
        xn,yn,xx,yx = src.bounds
        pol_str = "POLYGON((" + str(xn) + " " + str(yx) + ", " + \
                                str(xx) + " " + str(yx) + ", " + \
                                str(xx) + " " + str(yn) + ", " + \
                                str(xn) + " " + str(yn) + ", " + \
                                str(xn) + " " + str(yx) + "))"

        #qry = """
        #SELECT ST_GeometryType(a.geometry),*
        #    FROM public.buildings a
        #    WHERE ST_Intersects( ST_Transform(a.geometry,32633), 
        #                        ST_GeomFromText('""" + pol_str + """',32633))
        #"""
        qry = """
        SELECT ST_Intersection(ST_Transform(a.geometry,32633), b.geometry) as geometry
            FROM public.buildings a JOIN ST_GeomFromText('""" + pol_str + """',32633) as b
            ON ST_Intersects(ST_Transform(a.geometry,32633), b.geometry)
            ORDER BY a.index
        """
        

        res = gpd.read_postgis(qry,con,geom_col='geometry')
        if len(res)==0:
            print('  > None')
            continue
        else:
            # skip CRS transformation, it is already in 32633 projection:
            #r = res.to_crs(32633)
            
            # remove duplicate geometries:
            uGidx = get_unique_geom_indexes(res.geometry)
            print( '  > %d of %d duplicate geometries' % (res.shape[0]-len(uGidx),res.shape[0]) )
            r = res.iloc[uGidx]

            
            with open(ftxt,'w') as txt:
                for ir, row in r.iterrows():
                    if row.geometry.type=='Polygon':
                        x,y = np.array(row.geometry.exterior.coords.xy)
                        xr = (np.array(x) - xn) / (xx-xn)
                        yr = (yx - np.array(y)) / (yx-yn)

                        txt_line = '0 '
                        for j in range(len(xr)-1):
                            txt_line = txt_line + " " + str(xr[j]) + " " + str(yr[j])
                        txt.write(txt_line + "\n")
                        Count = Count +1

                    if row.geometry.type=='MultiPolygon':
                        re = row.explode()
                        for g in re.geometry:
                            x,y = np.array(g.exterior.coords.xy)
                            xr = (np.array(x) - xn) / (xx-xn)
                            yr = (yx - np.array(y)) / (yx-yn)            
                            txt_line = '0 '
                            for j in range(len(xr)-1):
                                txt_line = txt_line + " " + str(xr[j]) + " " + str(yr[j])
                            txt.write(txt_line + "\n") 
                            Count = Count +1
                                
            print('  > %d annotations' % r.shape[0])
            
print('\nTOT annotations:\n%d.\n' % Count)

warnings.resetwarnings()

#### Count all labels

In [ ]:
Nlines = 0
Ntxt = len( glob.glob(dir_annotation_ras + '*.txt') )
Ntif = len( glob.glob(dir_annotation_ras + '*.tif') )
for f in glob.glob( dir_annotation_ras + '*.txt' ):
    with open(f,'r') as txt:
        Nlines = Nlines + len(txt.readlines())

print("Folder    : ",dir_annotation_ras)
print(" N lines  : ",Nlines)
print(" N files  : %d / %d [txt / tif]" % (Ntxt,Ntif))

#### Double-check annotation geometries

In [ ]:
N = len(glob.glob( dir_annotation_ras + '*.txt' ))
print("%d annotation text files found in %s." % (N,dir_annotation_ras))

In [ ]:
from impervious.config import pg_url  # connessione da .env
isel = [0,77,170]
pol2backtransform = 2 # first two annotated polygons from TXT

db_connection_url = pg_url()
con = create_engine(db_connection_url)

i=-1
for f in glob.glob( dir_annotation_ras + '*.txt' ):
    i = i +1
    nLine=0
    if i in isel:
        print("\n%04d\t%s" % (i,f.split("/")[2]))

        # load TIF bbox:
        ftif = f.split('.txt')[0] + '.tif'
        src = rio.open(ftif)
        xn,yn,xx,yx = src.bounds
        
        # retrieve Geometries within TIF bbox:
        pol_str = "POLYGON((" + str(xn) + " " + str(yx) + ", " + \
                                str(xx) + " " + str(yx) + ", " + \
                                str(xx) + " " + str(yn) + ", " + \
                                str(xn) + " " + str(yn) + ", " + \
                                str(xn) + " " + str(yx) + "))"
        qry = """
        SELECT ST_Intersection(ST_Transform(a.geometry,32633), b.geometry) as geometry
            FROM public.buildings a JOIN ST_GeomFromText('""" + pol_str + """',32633) as b
            ON ST_Intersects(ST_Transform(a.geometry,32633), b.geometry)
            ORDER BY a.index
        """
        res = gpd.read_postgis(qry,con,geom_col='geometry')        

        # for selected lines in TXT given 'isel' tile:
        with open(f,'r') as txt:
            for line in txt:
                nLine = nLine + 1
                if nLine <= pol2backtransform:
                    line = line.replace('\n','')
                    print('  > %s' % line)
                    
                    # back-transform annotations into geojson:
                    xy = line[3:].split(' ')
                    x,y = xy[::2], xy[1::2]
                    
                    xa = (np.array(x,dtype=float) * (xx-xn)) + xn
                    ya = yx - (np.array(y,dtype=float) * (yx-yn))
                    
                    pol_geom = Polygon(zip(xa, ya))
                    
                    # analyse difference:
                    difference = res.geometry[nLine-1] - pol_geom
                    print('    Difference is an empty Polygon: %s' % bool(difference.is_empty))
                    print("    Area:  %.3f - %.3f = %.3f" % (pol_geom.area,res.geometry[nLine-1].area,difference.area) )
                    
                    # save:
                    pol = gpd.GeoDataFrame(index=[0], crs='epsg:32633', geometry=[pol_geom])
                    pol.to_file( filename='annotations/pol_backtransformed/' + \
                                 'pol' + str(nLine) + '_back__' + f.split('.txt')[0].split('/')[2] + '.geojson', driver='GeoJSON')
                    res.iloc[[nLine-1]].to_file( filename='annotations/pol_backtransformed/' + \
                                 'pol' + str(nLine) + '_orig__' + f.split('.txt')[0].split('/')[2] + '.geojson', driver='GeoJSON')

                    print("")

In [ ]:
def plot_two_polygons(geom0,geom1):
    new_shape = so.unary_union([ geom0, geom1 ])
    fig, axs = plt.subplots()
    axs.set_aspect('equal', 'datalim')

    i=0
    for geom in new_shape.geoms:
        xs, ys = geom.exterior.xy
        print('%d - %s' % (i,geom) )
        axs.fill(xs, ys, alpha=0.5, fc='black', ec='gray')
        i=i+1

    plt.show()

In [ ]:
plot_two_polygons(res.geometry[0],res.geometry[1])

#### Double-check duplicated annotations

In [ ]:
fil_annotated="annotations/imp_class/labels/val/S2A_MSIL1C_20230427T095031_N0509_R079_T33TVF_20230427T115004__013_016.txt"
fil_annotated

In [ ]:
with open(fil_annotated,'r') as txt:
    Nlines = len(txt.readlines())
    print("Nlines: ",Nlines)

duplicates = pd.DataFrame(columns = ['First', 'Second'])
ii = 0
with open(fil_annotated,'r') as txt:
    for count1, line1 in tqdm(enumerate(txt)):
        if count1>=0:
            xy1 = line1[3:].split(' ')
            with open(fil_annotated,'r') as txt2: 
                for count2, line2 in enumerate(txt2):
                    if count2>=0:
                        xy2 = line2[3:].split(' ')
                        if len(xy1) == len(xy2):
                            if count1 != count2:
                                d = np.absolute(np.array(xy1,dtype=float) - np.array(xy2,dtype=float))
                                s = sum( d )
                                if s==0:
                                    ii = ii+1;
                                    duplicates = pd.concat([duplicates,pd.DataFrame([[count1,count2]],columns=['First','Second'])], ignore_index = True)
                                    #print(ii, count1, count2, " - sum=",s, " - list ", d)

# unable to remove duplicates since they are on two different columns:
duplicates.drop_duplicates(inplace=True)
print("Number of duplicate labels %d." % len(duplicates))